# Chemical Search API

Search compound details by structure identifiers using the production nmrXiv search endpoint.

This notebook targets `POST /api/v1/search` at `https://nmrxiv.org/api/documentation#/search/search`.


## Supported query types

Use only these lowercase `type` values in this notebook:

- `smiles`
- `inchi`
- `inchikey`


In [25]:
import os
from pprint import pprint

import pandas as pd
import requests
from dotenv import load_dotenv

load_dotenv()

# This notebook intentionally defaults to the production API because the referenced Swagger URL is on nmrxiv.org.
BASE_URL = os.getenv("NMRXIV_SEARCH_BASE_URL", "https://nmrxiv.org").rstrip("/")
API_BASE = f"{BASE_URL}/api"

session = requests.Session()
session.headers.update({
    "Accept": "application/json",
    "Content-Type": "application/json",
})

BASE_URL, API_BASE


('https://nmrxiv.org', 'https://nmrxiv.org/api')

In [26]:
VALID_SEARCH_TYPES = {"smiles", "inchi", "inchikey"}


def api_request(method, path, **kwargs):
    url = f"{API_BASE}{path}"
    response = session.request(method, url, timeout=30, **kwargs)
    print(f"{response.request.method} {response.url} -> {response.status_code}")
    try:
        payload = response.json()
    except ValueError:
        payload = response.text
    if not response.ok:
        pprint(payload)
        response.raise_for_status()
    return payload


def as_dataframe(payload):
    if isinstance(payload, list):
        return pd.json_normalize(payload)
    if isinstance(payload, dict):
        for key in ["data", "results", "items"]:
            if isinstance(payload.get(key), list):
                return pd.json_normalize(payload[key])
        return pd.json_normalize(payload)
    return pd.DataFrame({"value": [payload]})


def search_compound(query, query_type):
    if query_type not in VALID_SEARCH_TYPES:
        valid = ", ".join(sorted(VALID_SEARCH_TYPES))
        raise ValueError(f"Unsupported query_type {query_type!r}. Use one of: {valid}")
    return api_request("POST", "/v1/search", json={"query": query, "type": query_type})


def result_summary(payload):
    if not isinstance(payload, dict):
        return {"total": None, "returned": None}
    data = payload.get("data", [])
    return {"total": payload.get("total"), "returned": len(data) if isinstance(data, list) else None}


## Search by SMILES

Use the lowercase query type `smiles`.


In [27]:
smiles_query = "CC(C)=C1CC[C@@H](C)CC1=O"
smiles_results = search_compound(smiles_query, "smiles")
result_summary(smiles_results)


POST https://nmrxiv.org/api/v1/search -> 200


{'total': 2, 'returned': 1}

In [28]:
as_dataframe(smiles_results).head()


,id,cas,molecular_formula,molecular_weight,smiles,absolute_smiles,canonical_smiles,inchi,standard_inchi,inchi_key,...,synonyms,iupac_name,2d,3d,structural_comments,status,active,has_stereo,has_variants,variants_count
0,183,None,C10H16O,152.23,None,None,CC(C)=C1CC[C@@H](C)CC1=O,None,InChI=1S/C10H16O/c1-7(2)9-5-4-8(3)6-10(9)11/h8...,NZGWDASTMWDZIW-MRVPVSSYSA-N,...,"(+)-Pulegone,Pulegone,89-82-7,d-Pulegone,Puleg...",(5R)-5-methyl-2-propan-2-ylidenecyclohexan-1-one,None,None,None,APPROVED,True,False,False,0


## Search by InChI

Use the lowercase query type `inchi` and pass the full `InChI=...` string.


In [29]:
inchi_query = "InChI=1S/C10H16O/c1-7(2)9-5-4-8(3)6-10(9)11/h8H,4-6H2,1-3H3/t8-/m1/s1"
inchi_results = search_compound(inchi_query, "inchi")
result_summary(inchi_results)


POST https://nmrxiv.org/api/v1/search -> 200


{'total': 1, 'returned': 1}

In [30]:
as_dataframe(inchi_results).head()


,id,cas,molecular_formula,molecular_weight,smiles,absolute_smiles,canonical_smiles,inchi,standard_inchi,inchi_key,...,synonyms,iupac_name,2d,3d,structural_comments,status,active,has_stereo,has_variants,variants_count
0,183,None,C10H16O,152.23,None,None,CC(C)=C1CC[C@@H](C)CC1=O,None,InChI=1S/C10H16O/c1-7(2)9-5-4-8(3)6-10(9)11/h8...,NZGWDASTMWDZIW-MRVPVSSYSA-N,...,"(+)-Pulegone,Pulegone,89-82-7,d-Pulegone,Puleg...",(5R)-5-methyl-2-propan-2-ylidenecyclohexan-1-one,None,None,None,APPROVED,True,False,False,0


## Search by InChIKey

Use the lowercase query type `inchikey`. Some valid InChIKey values may still return zero rows; that means no match was found by this endpoint.


In [35]:
inchikey_query = "NZGWDASTMWDZIW-MRVPVSSYSA-N"
inchikey_results = search_compound(inchikey_query, "inchikey")
result_summary(inchikey_results)


POST https://nmrxiv.org/api/v1/search -> 200


{'total': 0, 'returned': 0}

In [36]:
as_dataframe(inchikey_results).head()


""
